# Notebook 02 : indexation du corpus, des passages aux vecteurs servis au navigateur

**Ticket #30.** Ce notebook transforme les passages produits par le notebook 01 en vecteurs, vérifie que le navigateur
retrouvera les mêmes résultats que Python, et exporte l'index du site ainsi que les index destinés à l'évaluation.

**Le problème à résoudre.** Le site du volet A doit répondre à une question sans serveur de calcul : tout se passe
dans le navigateur du visiteur. Le calcul lourd, encoder les 4 240 passages, se fait donc une seule fois, ici, hors ligne. Le
navigateur n'aura plus qu'à encoder la question, la comparer à tous les vecteurs et afficher les meilleurs passages.
Tout l'enjeu est que ces deux calculs, faits sur deux machines et par deux logiciels différents, restent compatibles.

**Rappel : la recherche dense avec un bi-encodeur.** Un encodeur transforme un texte en un vecteur de nombres (ici 384),
appris de sorte que deux textes de sens proche aient des vecteurs proches. Dans un *bi-encodeur*, la question et les
passages sont encodés séparément, par le même modèle ; la proximité se mesure ensuite par le cosinus entre vecteurs.
Contrairement à une recherche par mots-clés (TF-IDF, BM25), un passage peut être retrouvé sans partager un seul mot
avec la question. Le modèle retenu au jalon J2 est `multilingual-e5-small` ; le support
`docs/etudes/indexation/01_encodeur_e5_small.pdf` détaille son fonctionnement.

| Entrée | Sortie |
|---|---|
| `data/decoupage/passages/` : tables de passages et manifeste du notebook 01 | `data/indexation/site/` : index du site (découpage M2), métadonnées et en-tête |
| modèles figés : `intfloat/multilingual-e5-small` (référence PyTorch) et `Xenova/multilingual-e5-small` (ONNX, servi au navigateur) | `data/indexation/variantes/` : un index par découpage, pour le notebook 03 |
| | `data/indexation/controle/` : requêtes et résultats des contrôles dans le navigateur |

**Le cheminement, en quatre temps**

| Temps | Étapes | Question traitée |
|---|---|---|
| Préparer | 1 à 3 | les entrées sont-elles intactes, et quels modèles utilise-t-on ? |
| Contrôler le calcul | 4 à 8 | le modèle du navigateur calcule-t-il comme la référence, et que coûte la quantification ? |
| Exporter | 9 et 10 | sous quelle forme livrer l'index au site et au notebook 03 ? |
| Vérifier l'export | 11 à 13 | le site retrouve-t-il exactement ce que Python calcule ? |

**Ce que ce notebook établit** : l'intégrité des entrées, la parité entre le modèle de référence et le modèle ONNX,
l'effet de la quantification sur le classement des passages, mesuré en Python puis **dans le navigateur**, le choix de
la précision servie au site, et l'intégrité de l'export vérifiée par le navigateur lui-même.

**Ce qu'il n'établit pas** : la qualité de la recherche. Elle exige le jeu de questions annotées, en préparation, et
fera l'objet du notebook 03, présenté à la fin. Les requêtes utilisées ici servent à **comparer des précisions de
calcul**, pas à juger des résultats.

**Comment lire ce notebook.** Même plan que le notebook 01 : pour chaque étape, son objectif, pourquoi et comment,
parfois un rappel théorique, le code commenté, puis une conclusion, les points à retenir pour la soutenance et la
transition vers l'étape suivante. Chaque critère de contrôle est écrit **avant** la mesure.

**Sommaire**
1. Configuration et reproductibilité
2. Entrées : intégrité des passages du notebook 01
3. Modèles : référence PyTorch et modèles ONNX du navigateur
4. Contrôle 1 : parité entre le modèle ONNX et la référence
5. Requêtes de contrôle
6. Contrôle 2 : effet de la quantification q8 sur le classement
7. Encodage des découpages
8. Contrôle 3 : fidélité de l'encodeur dans le navigateur, et choix de la précision servie
9. Export de l'index du site
10. Export des index d'évaluation
11. Exemples de recherche
12. Contrôle 4 : intégrité de l'export, vérifiée dans le navigateur
13. Manifeste et synthèse

## 1. Configuration et reproductibilité

**Objectif.** Exécuter le notebook dans l'environnement figé du dépôt, et n'écrire que dans `data/indexation/`, que git
ignore.

**Pourquoi.** Les vecteurs dépendent des versions exactes de PyTorch, d'ONNX Runtime et du tokeniseur : une autre
version peut changer les derniers chiffres, donc parfois un classement. Et les index produits ne doivent pas entrer
dans le dépôt sans décision de l'équipe.

**Comment.** Les mêmes conventions que le notebook 01. Le fichier figé de l'indexation contient toutes les dépendances
du notebook 01, plus ONNX Runtime : un seul environnement exécute les deux notebooks. Depuis la racine du dépôt :

```
uv venv --seed --python 3.12 .venv
uv pip sync scripts/indexation/requirements-lock.txt
```

Sans autre environnement activé, uv trouve seul le `.venv` du dossier courant : la commande est la même sous macOS,
Linux et Windows. Pour désigner l'interpréteur avec `--python`, il se trouve dans `.venv/bin/python` sous macOS et
Linux, et dans `.venv\Scripts\python.exe` sous Windows.

Dans VS Code : *Select Kernel*, *Python Environments*, `.venv`. Le premier lancement télécharge les deux fichiers ONNX
(448,5 et 112,8 Mio) et le modèle de référence (470 Mio) dans le cache Hugging Face, hors du dépôt. Les sorties vont
dans `data/indexation/`, que git ignore : la cellule le vérifie avant d'écrire.

In [1]:
import os
import warnings

# barres de progression des bibliothèques Hugging Face : sans intérêt ici, et bruyantes sans ipywidgets
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
warnings.filterwarnings("ignore", message="IProgress not found")

import gzip
import hashlib
import json
import platform
import subprocess
import sys
import time
from datetime import datetime, timezone
from math import comb
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
import sentence_transformers
import tokenizers
import torch
from huggingface_hub import HfApi, hf_hub_download
from sentence_transformers import SentenceTransformer
from tokenizers import Tokenizer
from transformers.utils import logging as journal_transformers

journal_transformers.disable_progress_bar()

try:
    display
except NameError:  # exécution hors Jupyter : affichage texte
    def display(objet):
        """Affichage de repli quand le code tourne hors de Jupyter."""
        print(objet.to_string() if isinstance(objet, pd.DataFrame) else objet)

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)
GRAINE = 20260927
torch.manual_seed(GRAINE)


def racine_depot(depart: Path) -> Path:
    """Remonte les dossiers parents jusqu'à la racine du dépôt git, celle qui contient .git."""
    for dossier in (depart, *depart.parents):
        if (dossier / ".git").exists():
            return dossier
    raise FileNotFoundError(f"aucun dépôt git au-dessus de {depart}")


# tous les chemins partent de la racine du dépôt, quel que soit le dossier où le notebook est lancé
RACINE = racine_depot(Path.cwd().resolve())
ENTREES = RACINE / "data" / "decoupage" / "passages"
SORTIES = RACINE / "data" / "indexation"
SITE, VARIANTES, CONTROLE, CACHE = (SORTIES / d for d in ("site", "variantes", "controle", "cache"))
for dossier in (SITE, VARIANTES, CONTROLE, CACHE):
    dossier.mkdir(parents=True, exist_ok=True)
# refus d'écrire si data/indexation n'est pas ignoré par git
ignore = subprocess.run(["git", "-C", str(RACINE), "check-ignore", "-q", str(SORTIES)]).returncode == 0
assert ignore, "data/indexation n'est pas ignoré par git : corriger l'exclusion avant d'aller plus loin"

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "torch": torch.__version__, "sentence-transformers": sentence_transformers.__version__,
            "onnxruntime": ort.__version__, "tokenizers": tokenizers.__version__}
print("interpréteur :", sys.executable)
assert sys.version_info[:2] == (3, 12), f"Python {platform.python_version()} : le notebook est validé en Python 3.12"
if Path(sys.prefix).resolve() != (RACINE / ".venv").resolve():
    print("attention : l'interpréteur n'est pas le .venv du dépôt ; les versions peuvent différer du fichier figé")
print("racine du dépôt :", RACINE)
print(VERSIONS)

interpréteur : /Users/mahe/Desktop/M1_DTA/NLP_PROJET/data_tigers_nlp/.venv/bin/python
racine du dépôt : /Users/mahe/Desktop/M1_DTA/NLP_PROJET/data_tigers_nlp
{'python': '3.12.11', 'numpy': '2.5.3', 'pandas': '3.0.6', 'torch': '2.14.0', 'sentence-transformers': '6.1.0', 'onnxruntime': '1.30.0', 'tokenizers': '0.23.2'}


**Paramètres.** Tous réunis ici. Les deux préfixes sont ceux du modèle E5 : `passage: ` pour les passages, `query: `
pour les questions ; le modèle a été entraîné ainsi, et sa carte prévient qu'on perd en qualité sans eux. Le seuil de
parité et les critères des contrôles dans le navigateur sont fixés **avant** toute mesure, pour qu'aucun seuil ne soit
choisi après coup au vu des résultats.

In [2]:
PREFIXE_PASSAGE = "passage: "
PREFIXE_REQUETE = "query: "
BUDGET = 512                    # longueur maximale de l'encodeur, tokens spéciaux compris
DIMENSION = 384
TAILLE_LOT = 32
SEUIL_PARITE = 0.99999          # contrôle 1 : cosinus minimal entre ONNX fp32 et la référence
N_REQUETES_NAVIGATEUR = 20      # contrôle 4 : titres-questions tirés au hasard, en plus des 4 requêtes d'exemple
TOLERANCE_EGALITE = 1e-6        # contrôle 4 : deux scores plus proches que cela sont des ex aequo
PRECISIONS_8_BITS = {"q8": "onnx/model_quantized.onnx", "int8": "onnx/model_int8.onnx", "uint8": "onnx/model_uint8.onnx"}
MODELE_REF = {"depot": "intfloat/multilingual-e5-small", "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3"}
MODELE_ONNX = {"depot": "Xenova/multilingual-e5-small", "revision": "761b726dd34fb83930e26aab4e9ac3899aa1fa78",
               "fichiers": {"fp32": "onnx/model.onnx", "q8": "onnx/model_quantized.onnx"}}
# découpages à indexer : table du notebook 01 et colonne à encoder
DECOUPAGES = {
    "m2": ("m2", "entree_encodeur"),                    # méthode retenue, index du site
    "m2_sans_entete": ("m2", "entree_sans_entete"),     # mesure de l'effet de l'en-tête
    "m2_256": ("m2_256", "entree_encodeur"),            # mesure de l'effet du budget
    "m0": ("m0", "entree_encodeur"),                    # référence : découpage de l'éditeur
    "m1": ("m1", "entree_encodeur"),                    # fenêtre fixe
    "m3": ("m3", "entree_encodeur"),                    # découpage sémantique
}
DECOUPAGE_SITE = "m2"
REQUETES_EXEMPLE = [
    "Quelles mentions sont interdites sur le bulletin de paie ?",
    "Mon employeur peut-il me licencier pendant un arrêt maladie ?",
    "Combien de temps peut durer la période d'essai d'un CDI ?",
    "Peut-on être viré quand on est en arrêt de travail ?",
]

**Conclusion.** L'environnement est celui du dépôt, les sorties iront dans un dossier ignoré par git, et tous les
réglages, seuils de décision compris, sont écrits avant la première mesure.

**Pour la soutenance.**

- Mêmes conventions que le notebook 01 : chemins depuis la racine, données hors de git, interpréteur affiché.
- Les seuils de décision sont écrits avant les mesures : on ne peut pas les ajuster pour que « ça passe ».
- Les deux modèles sont figés par leur révision : un autre commit du Hub donnerait d'autres vecteurs.

**Transition.** Avant d'encoder quoi que ce soit, il faut s'assurer que les passages sont bien ceux que le notebook 01 a
produits (étape 2).

## 2. Entrées : intégrité des passages du notebook 01

**Objectif.** Vérifier que les tables à encoder sont exactement celles que décrit le manifeste du notebook 01.

**Pourquoi.** Un index n'a de sens que rattaché à un découpage précis. Si une table avait été régénérée ou modifiée à
la main, les vecteurs ne correspondraient plus aux passages affichés, sans que rien ne le signale.

**Comment.** Le notebook 02 ne recompose rien : il encode la colonne `entree_encodeur` telle que le notebook 01 l'a
écrite. Avant d'encoder, il vérifie que les fichiers sont bien ceux que décrit le manifeste du notebook 01 (empreinte
du fichier et empreinte du contenu), que chaque entrée porte le préfixe `passage: `, et que le nombre de tokens,
**recompté ici** avec le tokeniseur du modèle servi, respecte la limite de 512.

In [3]:
def sha256_fichier(chemin: Path) -> str:
    """Empreinte SHA-256 d'un fichier, lue par blocs de 1 Mio."""
    h = hashlib.sha256()
    with open(chemin, "rb") as f:
        for bloc in iter(lambda: f.read(1 << 20), b""):
            h.update(bloc)
    return h.hexdigest()


def empreinte_contenu(ids, entrees) -> str:
    """Empreinte du contenu encodable, calculée exactement comme au notebook 01."""
    h = hashlib.sha256()
    for pid, entree in zip(ids, entrees):
        h.update(f"{pid}\x1f{entree}\x1e".encode())
    return h.hexdigest()


# le manifeste du notebook 01 donne, pour chaque table, l'empreinte du fichier et celle du contenu
manifeste01 = json.loads((ENTREES / "manifeste.json").read_text(encoding="utf-8"))
TABLES, lignes = {}, []
for code in sorted({table for table, _ in DECOUPAGES.values()}):
    decrit = manifeste01["methodes"][code]
    chemin = RACINE / decrit["fichier"]
    assert sha256_fichier(chemin) == decrit["sha256_fichier"], f"{code} : fichier différent de celui du manifeste"
    P = pd.read_parquet(chemin)
    assert empreinte_contenu(P.passage_id, P.entree_encodeur) == decrit["empreinte_contenu"], f"{code} : contenu altéré"
    assert P.passage_id.is_unique and len(P) == decrit["passages"]
    TABLES[code] = P
    lignes.append({"table": code, "passages": len(P), "empreinte du fichier": "conforme", "empreinte du contenu": "conforme"})
display(pd.DataFrame(lignes))

# recomptage indépendant des tokens, avec le tokeniseur du modèle servi au navigateur
tok = Tokenizer.from_pretrained(MODELE_ONNX["depot"], revision=MODELE_ONNX["revision"])
lignes = []
for nom, (table, colonne) in DECOUPAGES.items():
    textes = TABLES[table][colonne]
    assert textes.str.startswith(PREFIXE_PASSAGE).all(), f"{nom} : préfixe manquant"
    assert (textes.str.len() > len(PREFIXE_PASSAGE)).all(), f"{nom} : entrée vide"
    n = np.array([len(e.ids) for e in tok.encode_batch(textes.tolist())])
    if colonne == "entree_encodeur":
        assert (n == TABLES[table].n_tokens.to_numpy()).all(), f"{nom} : comptage différent de celui du notebook 01"
    assert n.max() <= BUDGET, f"{nom} : {n.max()} tokens, au-delà de la limite"
    lignes.append({"découpage": nom, "passages": len(textes), "tokens moyenne": round(n.mean(), 1), "max": int(n.max())})
display(pd.DataFrame(lignes))

,table,passages,empreinte du fichier,empreinte du contenu
0,m0,5284,conforme,conforme
1,m1,5317,conforme,conforme
2,m2,4240,conforme,conforme
3,m2_256,7009,conforme,conforme
4,m3,4229,conforme,conforme


,découpage,passages,tokens moyenne,max
0,m2,4240,259.1,512
1,m2_sans_entete,4240,228.5,501
2,m2_256,7009,174.1,256
3,m0,5284,253.1,440
4,m1,5317,258.7,329
5,m3,4229,260.8,493


**Conclusion.** Les cinq tables sont conformes au manifeste, empreinte du fichier et empreinte du contenu, et le
recomptage des tokens redonne exactement celui du notebook 01. Aucune entrée ne dépasse 512 tokens : l'encodeur ne
tronquera aucun passage.

**Pour la soutenance.**

- Le notebook 02 part de fichiers dont l'empreinte est vérifiée : on sait qu'il encode exactement ce que le 01 a produit.
- Le nombre de tokens est recompté indépendamment, et aucune entrée ne dépasse 512 : l'encodeur ne tronquera rien.

**Transition.** Les entrées sont sûres. Il faut maintenant charger les modèles, et comprendre pourquoi il en faut deux
versions (étape 3).

## 3. Modèles : référence PyTorch et modèles ONNX du navigateur

**Objectif.** Charger le même modèle sous ses deux formes : celle de référence, en Python, et celle que le navigateur
exécutera.

**Pourquoi.** Le navigateur ne sait pas exécuter PyTorch. Il charge des fichiers ONNX, par la bibliothèque
Transformers.js. Pour que l'index calculé ici soit utilisable là-bas, il faut pouvoir reproduire en Python le calcul
exact du navigateur, et le comparer à la référence.

**Comment.** Deux familles de fichiers pour un même modèle :

- **la référence** : `intfloat/multilingual-e5-small` exécuté par PyTorch en fp32, via sentence-transformers, qui
  applique la moyenne des tokens puis la normalisation ;
- **les fichiers ONNX** de `Xenova/multilingual-e5-small`, ceux que Transformers.js charge dans le navigateur :
  `model.onnx` (fp32) et `model_quantized.onnx` (q8, chargé par défaut en WASM). Leurs empreintes sont comparées à
  celles que publie le Hub.

**Rappel : ONNX et ONNX Runtime.** ONNX (*Open Neural Network Exchange*) est un format ouvert qui décrit un modèle
comme un graphe d'opérations (produits de matrices, additions, normalisations), avec ses poids. ONNX Runtime est le
moteur qui exécute ce graphe : en Python sur le processeur, ou dans le navigateur par ONNX Runtime Web, compilé en
WebAssembly. Un même fichier ONNX peut donc être exécuté ici et chez le visiteur.

**Rappel : d'un vecteur par token à un vecteur par texte.** Les fichiers ONNX ne renvoient qu'un vecteur par token
(`last_hidden_state`), noté $\mathbf{h}_t$. Pour obtenir un vecteur par texte, on fait la **moyenne des vecteurs des
tokens réels**, en ignorant les tokens de remplissage grâce au masque d'attention $m_t$ (1 pour un token réel, 0 pour
un remplissage), puis on **normalise** le résultat :

$$\mathbf{e} = \frac{\sum_t m_t\,\mathbf{h}_t}{\sum_t m_t}, \qquad \hat{\mathbf{e}} = \frac{\mathbf{e}}{\lVert \mathbf{e} \rVert}$$

La fonction `encoder_onnx` reproduit ce que fait Transformers.js avec les options `{ pooling: "mean", normalize: true }`,
en float32, comme dans le navigateur. Après normalisation, tous les vecteurs ont une norme de 1 : le cosinus entre deux
vecteurs se réduit à leur produit scalaire.

In [4]:
ref = SentenceTransformer(MODELE_REF["depot"], revision=MODELE_REF["revision"], device="cpu")
print(ref)

# empreintes publiées par le Hub pour chaque fichier de la révision figée
hub = HfApi().model_info(MODELE_ONNX["depot"], revision=MODELE_ONNX["revision"], files_metadata=True)
empreintes_hub = {s.rfilename: s.lfs.sha256 for s in hub.siblings if s.lfs}
SESSIONS, FICHIERS_ONNX = {}, {}
for precision, fichier in MODELE_ONNX["fichiers"].items():
    chemin = Path(hf_hub_download(MODELE_ONNX["depot"], fichier, revision=MODELE_ONNX["revision"]))
    empreinte = sha256_fichier(chemin)
    assert empreinte == empreintes_hub[fichier], f"{fichier} : empreinte différente de celle du Hub"
    SESSIONS[precision] = ort.InferenceSession(str(chemin), providers=["CPUExecutionProvider"])
    FICHIERS_ONNX[precision] = {"fichier": fichier, "sha256": empreinte, "taille_Mio": round(chemin.stat().st_size / 2**20, 1)}
    entrees = [i.name for i in SESSIONS[precision].get_inputs()]
    sorties = [o.name for o in SESSIONS[precision].get_outputs()]
    print(f"{precision:5} {fichier:28} {FICHIERS_ONNX[precision]['taille_Mio']:6.1f} Mio  entrées {entrees}  sorties {sorties}")
assert entrees == ["input_ids", "attention_mask", "token_type_ids"] and sorties == ["last_hidden_state"]
# token de remplissage : il complète les textes courts d'un lot jusqu'à la longueur du plus long
ID_REMPLISSAGE = tok.token_to_id("<pad>")
print("identifiant du token de remplissage :", ID_REMPLISSAGE)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


fp32  onnx/model.onnx               448.5 Mio  entrées ['input_ids', 'attention_mask', 'token_type_ids']  sorties ['last_hidden_state']
q8    onnx/model_quantized.onnx     112.8 Mio  entrées ['input_ids', 'attention_mask', 'token_type_ids']  sorties ['last_hidden_state']
identifiant du token de remplissage : 1


In [5]:
def encoder_ref(textes: list) -> np.ndarray:
    """Encodage de référence : PyTorch fp32, moyenne des tokens et normalisation par sentence-transformers."""
    return ref.encode(list(textes), batch_size=TAILLE_LOT, normalize_embeddings=True,
                      convert_to_numpy=True, show_progress_bar=False).astype(np.float32)


def encoder_onnx(textes: list, precision: str) -> np.ndarray:
    """Encodage par un fichier ONNX, comme Transformers.js : tokenisation, moyenne masquée, normalisation L2."""
    session = SESSIONS[precision]
    encs = tok.encode_batch(list(textes))
    longueurs = np.array([len(e.ids) for e in encs])
    assert longueurs.max() <= BUDGET, "une entrée dépasse la limite : elle serait tronquée"
    ordre = np.argsort(longueurs, kind="stable")      # lots de longueurs voisines : moins de remplissage
    sortie = np.empty((len(encs), DIMENSION), dtype=np.float32)
    for debut in range(0, len(ordre), TAILLE_LOT):
        lot = ordre[debut:debut + TAILLE_LOT]
        L = int(longueurs[lot].max())
        ids = np.full((len(lot), L), ID_REMPLISSAGE, dtype=np.int64)
        masque = np.zeros((len(lot), L), dtype=np.int64)
        for k, i in enumerate(lot):
            ids[k, :longueurs[i]] = encs[i].ids
            masque[k, :longueurs[i]] = 1
        # passage dans le modèle ONNX : un vecteur de 384 nombres par token, pour chaque texte du lot
        etats = session.run(None, {"input_ids": ids, "attention_mask": masque,
                                   "token_type_ids": np.zeros_like(ids)})[0]            # (lot, L, 384)
        m = masque[..., None].astype(np.float32)
        moyenne = (etats * m).sum(axis=1) / m.sum(axis=1)                                # moyenne des tokens réels
        sortie[lot] = moyenne / np.linalg.norm(moyenne, axis=1, keepdims=True)          # normalisation L2
    return sortie


def encoder(textes: list, moteur: str, nom: str) -> np.ndarray:
    """Encode avec le moteur demandé (ref, fp32 ou q8), en réutilisant un cache dont la clé dépend de tout ce qui fixe le résultat."""
    textes = list(textes)
    revision = MODELE_REF["revision"] if moteur == "ref" else FICHIERS_ONNX[moteur]["sha256"]
    # textes, moteur, révision ou fichier du modèle, taille de lot (le remplissage d'un lot change les derniers
    # chiffres) et versions des bibliothèques : un changement de l'un d'eux invalide le cache
    contexte = f"{moteur}|{revision}|{TAILLE_LOT}|{json.dumps(VERSIONS, sort_keys=True)}"
    cle = hashlib.sha256(("\x1e".join(textes) + "\x1d" + contexte).encode()).hexdigest()[:16]
    fichier = CACHE / f"{nom}_{moteur}_{cle}.npy"
    duree = fichier.with_suffix(".json")      # durée du calcul d'origine, conservée pour le manifeste
    if fichier.exists() and duree.exists():
        DUREES[(nom, moteur)] = json.loads(duree.read_text())["secondes"]
        return np.load(fichier)
    debut = time.perf_counter()
    E = encoder_ref(textes) if moteur == "ref" else encoder_onnx(textes, moteur)
    DUREES[(nom, moteur)] = time.perf_counter() - debut
    np.save(fichier, E)
    duree.write_text(json.dumps({"secondes": DUREES[(nom, moteur)], "textes": len(textes)}))
    return E


DUREES = {}

**Conclusion.** Les deux fichiers ONNX ont l'empreinte publiée par le Hub : 448,5 Mio en fp32, 112,8 Mio en q8. Ils
attendent les mêmes entrées (`input_ids`, `attention_mask`, `token_type_ids`) et renvoient `last_hidden_state`. Les
fonctions d'encodage sont prêtes, avec un cache qui évite de refaire un calcul déjà fait. L'avertissement
« unauthenticated requests to the HF Hub » affiché en rouge est sans conséquence : les deux dépôts sont publics, et un
jeton ne ferait qu'élever la limite de téléchargements.

**Pour la soutenance.**

- Le navigateur n'exécute pas PyTorch mais des fichiers ONNX : il faut vérifier que les deux calculent la même chose.
- Les fichiers ONNX sont contrôlés par l'empreinte publiée sur le Hub.
- `encoder_onnx` reproduit ligne à ligne la moyenne masquée et la normalisation que Transformers.js applique.

**Transition.** Encore faut-il prouver que `encoder_onnx` calcule bien comme la référence : c'est le contrôle 1
(étape 4).

## 4. Contrôle 1 : parité entre le modèle ONNX et la référence

**Objectif.** Vérifier que le fichier `model.onnx` (fp32), exécuté par notre fonction `encoder_onnx`, donne les mêmes
vecteurs que la référence PyTorch.

**Pourquoi.** Si c'est le cas, notre reproduction du calcul du navigateur est juste, et toute différence observée
ensuite avec le q8 viendra de la quantification seule. Sinon, tous les contrôles suivants seraient faussés.

**Comment.** Les 4 240 passages M2 sont tokenisés par les deux tokeniseurs, puis encodés par les deux moteurs ; on
compare les vecteurs un à un.

**Critères fixés à l'avance.** Tokenisation identique pour 100 % des entrées ; cosinus minimal d'au moins 0,99999
entre les deux encodages de chaque passage M2.

**Rappel : pourquoi pas une égalité parfaite.** Un nombre float32 a 24 bits de mantisse, soit environ 7 chiffres
décimaux significatifs. Deux logiciels qui additionnent les mêmes nombres dans un ordre différent obtiennent des
résultats qui diffèrent sur les derniers chiffres. On attend donc des vecteurs égaux à environ 1e-7 près, pas des
vecteurs identiques bit à bit : d'où un seuil de 0,99999, et non de 1.

In [6]:
passages_site = TABLES["m2"].entree_encodeur.tolist()
ecarts_tok = [t for t in passages_site if ref.tokenizer(t)["input_ids"] != tok.encode(t).ids]
print(f"tokenisation : {len(passages_site) - len(ecarts_tok)} entrées identiques sur {len(passages_site)}")
assert not ecarts_tok

# les mêmes passages, encodés par les deux moteurs, puis comparés vecteur à vecteur
E_ref = encoder(passages_site, "ref", "m2")
E_fp32 = encoder(passages_site, "fp32", "m2")
# vecteurs normalisés : le produit scalaire ligne à ligne est le cosinus entre les deux encodages
cos_parite = np.einsum("ij,ij->i", E_ref, E_fp32)
print(f"cosinus ONNX fp32 et référence : minimum {cos_parite.min():.8f}, médiane {np.median(cos_parite):.8f} ; "
      f"écart absolu maximal sur une composante {np.abs(E_ref - E_fp32).max():.2e}")
assert cos_parite.min() >= SEUIL_PARITE, "parité non atteinte : la reproduction du calcul du navigateur est à revoir"

tokenisation : 4240 entrées identiques sur 4240
cosinus ONNX fp32 et référence : minimum 0.99999976, médiane 1.00000000 ; écart absolu maximal sur une composante 2.76e-07


**Conclusion.** Contrôle 1 réussi : tokenisation identique pour 4 240 passages sur 4 240, cosinus minimal de
0,99999976, et écart maximal sur une composante de l'ordre de 3e-7, soit la précision du float32.

**Pour la soutenance.**

- Deux implémentations indépendantes du même modèle donnent les mêmes vecteurs, aux arrondis de calcul près.
- La moyenne masquée et la normalisation de `encoder_onnx` sont donc justes ; le reste du notebook peut s'y fier.

**Transition.** Pour mesurer l'effet de la quantification sur le classement, il faut des requêtes. Le jeu de questions
n'existant pas encore, l'étape 5 en construit à partir des fiches elles-mêmes.

## 5. Requêtes de contrôle

**Objectif.** Disposer de centaines de requêtes réalistes pour comparer des calculs, sans attendre le jeu annoté.

**Pourquoi.** Comparer deux précisions de calcul demande beaucoup de requêtes, mais pas de vérité terrain : il suffit
de vérifier que deux calculs classent les passages de la même façon.

**Comment.** Le jeu de questions annotées n'existe pas encore. On prend les titres de section formulés en questions
(« Quelle est la durée de la période d'essai ? »), nombreux dans les fiches, préfixés par `query: `.

**Limite à ne pas oublier.** Ces titres figurent dans l'en-tête des passages M2 : la recherche les retrouve trop
facilement. Ils conviennent pour vérifier que deux calculs classent les passages de la même façon ; ils ne disent
**rien** de la qualité de la recherche, et ne servent jamais à comparer les découpages entre eux.

In [7]:
# titres de section qui sont des questions : des requêtes réalistes, sans rédaction humaine
TITRES_QUESTIONS = sorted({t.strip() for t in TABLES["m2"].section_titre if t.strip().endswith("?")})
REQUETES = [PREFIXE_REQUETE + t for t in TITRES_QUESTIONS]
print(f"{len(TITRES_QUESTIONS)} titres-questions distincts sur {TABLES['m2'].section_titre.nunique()} titres de section")
print("exemples :", TITRES_QUESTIONS[:3])
assert all(ref.tokenizer(q)["input_ids"] == tok.encode(q).ids for q in REQUETES)

# liste lue par la page de fidélité du navigateur (section 8)
(CONTROLE / "requetes_fidelite.json").write_text(
    json.dumps({"prefixe": PREFIXE_REQUETE, "requetes": TITRES_QUESTIONS}, ensure_ascii=False), encoding="utf-8")

Q_ref = encoder(REQUETES, "ref", "requetes")
Q_fp32 = encoder(REQUETES, "fp32", "requetes")
Q_q8 = encoder(REQUETES, "q8", "requetes")
cos_q = np.einsum("ij,ij->i", Q_ref, Q_fp32)
print(f"parité sur les requêtes : cosinus minimal {cos_q.min():.8f}")
assert cos_q.min() >= SEUIL_PARITE

822 titres-questions distincts sur 1301 titres de section
exemples : ['A quel moment doit intervenir l’examen médical ?', 'A quel âge peut-on travailler en France ?', 'A quelles exigences une commande bimanuelle doit-elle répondre ?']
parité sur les requêtes : cosinus minimal 0.99999976


**Conclusion.** 822 titres-questions distincts, sur 1 301 titres de section, servent de requêtes de contrôle. Le
contrôle de parité tient aussi sur elles : cosinus minimal de 0,99999976.

**Pour la soutenance.**

- En attendant le jeu annoté, les titres-questions donnent des centaines de requêtes réalistes, sans coût humain.
- Ils ne mesurent que l'accord entre deux calculs, jamais la qualité : leur texte figure dans l'en-tête des passages.

**Transition.** Le navigateur n'encodera pas la question en fp32 mais avec le modèle q8, quatre fois plus léger. Que
coûte cette compression au classement ? C'est le contrôle 2 (étape 6).

## 6. Contrôle 2 : effet de la quantification q8 sur le classement

**Objectif.** Mesurer ce que la quantification q8 change au classement des passages, et choisir comment calculer
l'index.

**Pourquoi.** Le modèle q8 pèse 112,8 Mio au lieu de 448,5 : c'est lui que le navigateur télécharge. Ses vecteurs sont
très proches de ceux du fp32, mais les scores des premiers passages sont souvent séparés de quelques millièmes : un
petit écart sur les vecteurs peut inverser deux résultats. Il faut donc mesurer l'effet sur le **classement**, pas
seulement sur les vecteurs.

**Rappel : la quantification.** Quantifier, c'est stocker un nombre réel $x$ sur 8 bits au lieu de 32, sous la forme
$x \approx s\,(q - z)$ : $q$ est un entier de 8 bits, $s$ une échelle et $z$ un décalage. La taille est divisée par
quatre, au prix d'une petite erreur d'arrondi sur chaque valeur. Le fichier q8 utilise une quantification
*dynamique* : les poids des produits de matrices sont stockés en entiers de 8 bits, et les valeurs intermédiaires sont
quantifiées à la volée, au moment du calcul. Inspecté avec la bibliothèque `onnx` (hors de ce notebook, dont elle n'est
pas une dépendance), son graphe contient 72 produits entiers `MatMulInteger`, soit 6 par couche sur 12 couches, et 48
opérations `DynamicQuantizeLinear` ; les poids des produits sont en `INT8`, la table d'embeddings en `UINT8`.

Dans le navigateur, la question sera encodée par le modèle q8. Deux façons de construire l'index sont possibles :

| Configuration | Index des passages | Requête | Rôle |
|---|---|---|---|
| référence | PyTorch fp32 | PyTorch fp32 | le classement sans aucune quantification |
| A | PyTorch fp32 | ONNX q8 | index calculé une fois au mieux, requête du navigateur |
| B | ONNX q8 | ONNX q8 | index et requête calculés par le même modèle quantifié |

Une troisième configuration, index ONNX fp32 et requête q8, est inutile : le contrôle 1 a montré que l'index ONNX
fp32 est identique à l'index PyTorch.

**Mesures.** Pour chaque configuration, comparée à la référence : part des requêtes dont le premier résultat est le
même, recouvrement moyen des 5 et des 10 premiers résultats, part des requêtes dont le premier résultat de référence
reste parmi les 5 premiers (ce que voit un visiteur à qui l'on affiche 5 résultats), et rang le plus bas auquel il
tombe.

**Règle de décision, fixée avant la mesure.** On retient A, la plus simple et indépendante de la quantification, sauf
si B reproduit significativement mieux le premier résultat de référence (test exact de McNemar sur les requêtes
discordantes, au seuil de 5 %).

**Rappel : le test exact de McNemar.** Les deux configurations sont jugées sur les **mêmes** requêtes : les mesures
sont appariées. Seules comptent les requêtes *discordantes* : $b$ celles où A seule retrouve le premier résultat de
référence, $c$ celles où B seule le retrouve. Si les deux configurations se valaient, chaque requête discordante
pencherait d'un côté ou de l'autre avec une probabilité de 1/2 : $b$ suivrait une loi binomiale de paramètres
$(b + c, 1/2)$. La valeur p bilatérale exacte vaut $\min\big(1,\ 2\,P(X \le \min(b, c))\big)$ pour cette loi. Une
valeur p inférieure à 0,05 signifie qu'un écart aussi net serait improbable si les deux configurations se valaient.

In [8]:
E_q8 = encoder(passages_site, "q8", "m2")
cos_quant = np.einsum("ij,ij->i", E_ref, E_q8)
print(f"passages, cosinus q8 et référence : minimum {cos_quant.min():.5f}, médiane {np.median(cos_quant):.5f}")
print(f"requêtes, cosinus q8 et référence : minimum {np.einsum('ij,ij->i', Q_ref, Q_q8).min():.5f}, "
      f"médiane {np.median(np.einsum('ij,ij->i', Q_ref, Q_q8)):.5f}")


def classement(Q: np.ndarray, E: np.ndarray, k: int = 10) -> tuple:
    """Scores complets et indices des k meilleurs passages pour chaque requête, par produit scalaire."""
    S = Q @ E.T
    return S, np.argsort(-S, axis=1, kind="stable")[:, :k]


def accord(top: np.ndarray, S: np.ndarray, top_ref: np.ndarray) -> dict:
    """Accord d'un classement avec celui de référence : premier résultat, recouvrements, rang du premier de référence."""
    rang_premier = np.array([(S[i] > S[i, top_ref[i, 0]]).sum() + 1 for i in range(len(S))])
    return {"même premier résultat %": round(100 * np.mean(top[:, 0] == top_ref[:, 0]), 1),
            "recouvrement du top 5 %": round(100 * np.mean([len(set(a[:5]) & set(b[:5])) / 5 for a, b in zip(top, top_ref)]), 1),
            "recouvrement du top 10 %": round(100 * np.mean([len(set(a) & set(b)) / 10 for a, b in zip(top, top_ref)]), 1),
            "premier de référence dans le top 5 %": round(100 * np.mean(rang_premier <= 5), 1),
            "rang du premier de référence, maximum": int(rang_premier.max())}


def mcnemar_exact(a_juste: np.ndarray, b_juste: np.ndarray) -> dict:
    """Test exact de McNemar, bilatéral : ne compte que les requêtes où une seule des deux configurations est juste."""
    b = int(np.sum(a_juste & ~b_juste))
    c = int(np.sum(~a_juste & b_juste))
    n = b + c
    # sous l'hypothèse nulle, b suit une loi binomiale (n, 1/2) : valeur p bilatérale exacte, plafonnée à 1
    p = 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n)
    return {"A juste seule": b, "B juste seule": c, "p": p}


# trois classements des 822 requêtes : référence, configuration A, configuration B
S_ref, top_ref = classement(Q_ref, E_ref)
S_A, top_A = classement(Q_q8, E_ref)
S_B, top_B = classement(Q_q8, E_q8)
tableau_quant = pd.DataFrame({"A : index fp32, requête q8": accord(top_A, S_A, top_ref),
                              "B : index q8, requête q8": accord(top_B, S_B, top_ref)}).T
display(tableau_quant)
test = mcnemar_exact(top_A[:, 0] == top_ref[:, 0], top_B[:, 0] == top_ref[:, 0])
print("test exact de McNemar sur le premier résultat :", test)
# règle écrite avant la mesure : B seulement si elle fait significativement mieux que A
CONFIGURATION = "B" if (test["p"] < 0.05 and test["B juste seule"] > test["A juste seule"]) else "A"
print("configuration retenue par la règle fixée à l'avance :", CONFIGURATION)

passages, cosinus q8 et référence : minimum 0.99817, médiane 0.99891
requêtes, cosinus q8 et référence : minimum 0.99825, médiane 0.99898


,même premier résultat %,recouvrement du top 5 %,recouvrement du top 10 %,premier de référence dans le top 5 %,"rang du premier de référence, maximum"
"A : index fp32, requête q8",96.5,96.0,95.7,100.0,2.0
"B : index q8, requête q8",93.2,94.5,94.0,100.0,4.0


test exact de McNemar sur le premier résultat : {'A juste seule': 38, 'B juste seule': 11, 'p': 0.00014197068519905542}
configuration retenue par la règle fixée à l'avance : A


**Stockage de l'index en entiers de 8 bits.** L'index du site pourrait lui aussi être livré en int8 (une échelle par
dimension) au lieu de fp32, soit quatre fois moins lourd (tailles affichées ci-dessous). On mesure ce que coûte cette
compression au classement, dans la configuration retenue.

In [9]:
E_config = E_ref if CONFIGURATION == "A" else E_q8
# quantification symétrique par dimension : la plus grande valeur absolue de chaque colonne devient 127
echelle = np.abs(E_config).max(axis=0) / 127.0
E_int8 = np.clip(np.round(E_config / echelle), -127, 127).astype(np.int8)
S_cfg, top_cfg = classement(Q_q8, E_config)
S_i8, top_i8 = classement(Q_q8, E_int8.astype(np.float32) * echelle)
display(pd.DataFrame({"index int8 contre index fp32 (même requête q8)": accord(top_i8, S_i8, top_cfg)}).T)
print(f"taille : fp32 {E_config.nbytes / 2**20:.2f} Mio, int8 {E_int8.nbytes / 2**20:.2f} Mio ; "
      f"le modèle q8 téléchargé par le navigateur pèse {FICHIERS_ONNX['q8']['taille_Mio']} Mio")

,même premier résultat %,recouvrement du top 5 %,recouvrement du top 10 %,premier de référence dans le top 5 %,"rang du premier de référence, maximum"
index int8 contre index fp32 (même requête q8),98.7,98.2,98.2,100.0,3.0


taille : fp32 6.21 Mio, int8 1.55 Mio ; le modèle q8 téléchargé par le navigateur pèse 112.8 Mio


**Limite découverte à l'exécution.** Ce contrôle simule le q8 avec ONNX Runtime en Python. La section 8 montre que le
navigateur exécute le même fichier q8 de façon moins fidèle : la simulation Python **sous-estime** l'erreur réelle du
site. La mesure qui fait foi pour le site est celle de la section 8, faite dans le navigateur.

**Conclusion.** Les vecteurs q8 sont très proches de la référence (cosinus minimal de 0,998 environ), mais le premier
résultat change pour une part des requêtes. La configuration A reproduit le premier résultat de référence pour 96,5 %
des requêtes, contre 93,2 % pour B ; McNemar compte 38 requêtes où seule A est juste contre 11 pour B, p = 0,00014 :
A est retenue. Stocker l'index en int8 changerait encore le premier résultat pour 1,3 % des requêtes, pour gagner
moins de 5 Mio : l'index reste en fp32.

**Pour la soutenance.**

- La quantification q8 ne change presque pas les vecteurs, mais le classement fin est sensible : on mesure donc
  l'effet sur les premiers résultats, pas seulement sur les vecteurs.
- La configuration est choisie par une règle écrite avant la mesure, avec un test apparié.
- L'index est livré en fp32 : l'int8 ferait gagner quelques Mio, négligeables face au modèle, au prix de premiers
  résultats modifiés.

**Transition.** La configuration A fixe le moteur des passages : PyTorch fp32. L'étape 7 encode avec lui tous les
découpages du notebook 01.

## 7. Encodage des découpages

**Objectif.** Encoder les six découpages : M2 pour le site, et les autres pour l'évaluation du notebook 03.

**Pourquoi.** Le notebook 03 comparera les découpages sur le jeu de questions. Encoder tout maintenant, avec le même
moteur et les mêmes contrôles, garantit que les écarts mesurés plus tard viendront des découpages, pas de l'encodage.

**Comment.** Tous les découpages sont encodés avec le moteur de la configuration retenue pour les passages : PyTorch
fp32 pour A, ONNX q8 pour B. Contrôles sur chaque matrice : dimension, valeurs finies, norme égale à 1, et deux entrées
identiques doivent donner deux vecteurs identiques. Enfin, un échantillon de passages est réencodé un par un : le
résultat ne doit pas dépendre du lot dans lequel un passage a été encodé.

**Rappel : lots et remplissage.** Pour aller vite, on encode les textes par lots de 32. Les textes d'un lot n'ont pas la
même longueur : les plus courts sont complétés par des tokens de remplissage, que le masque d'attention exclut du
calcul et de la moyenne. Un passage doit donc avoir le même vecteur, seul ou dans un lot, aux arrondis près.

In [10]:
MOTEUR_INDEX = "ref" if CONFIGURATION == "A" else "q8"
INDEX, lignes = {}, []
for nom, (table, colonne) in DECOUPAGES.items():
    textes = TABLES[table][colonne].tolist()
    E = E_config if nom == DECOUPAGE_SITE else encoder(textes, MOTEUR_INDEX, nom)
    normes = np.linalg.norm(E, axis=1)
    assert E.shape == (len(textes), DIMENSION) and E.dtype == np.float32 and np.isfinite(E).all(), nom
    assert np.abs(normes - 1).max() < 1e-5, f"{nom} : norme hors de 1 ± 1e-5"
    # deux entrées identiques doivent donner exactement le même vecteur
    doublons = pd.Series(range(len(textes))).groupby(pd.Series(textes)).apply(list)
    doublons = [ix for ix in doublons if len(ix) > 1]
    ecart_doublons = max((np.abs(E[ix] - E[ix[0]]).max() for ix in doublons), default=0.0)
    assert ecart_doublons < 1e-6, f"{nom} : deux entrées identiques ont des vecteurs différents"
    INDEX[nom] = E
    lignes.append({"découpage": nom, "passages": len(textes), "écart de norme maximal": f"{np.abs(normes - 1).max():.1e}",
                   "entrées en double": sum(len(ix) - 1 for ix in doublons), "taille fp32 Mio": round(E.nbytes / 2**20, 2)})
display(pd.DataFrame(lignes))

rng = np.random.default_rng(GRAINE)
# 48 passages réencodés seuls : leur vecteur doit être celui obtenu dans leur lot
echantillon = rng.choice(len(passages_site), 48, replace=False)
un_par_un = np.stack([(encoder_ref([passages_site[i]]) if MOTEUR_INDEX == "ref" else encoder_onnx([passages_site[i]], "q8"))[0]
                      for i in echantillon])
print(f"passage encodé seul ou dans un lot : écart maximal {np.abs(un_par_un - INDEX[DECOUPAGE_SITE][echantillon]).max():.2e}")
assert np.abs(un_par_un - INDEX[DECOUPAGE_SITE][echantillon]).max() < 1e-5

,découpage,passages,écart de norme maximal,entrées en double,taille fp32 Mio
0,m2,4240,1.8e-07,3,6.21
1,m2_sans_entete,4240,1.8e-07,57,6.21
2,m2_256,7009,1.8e-07,6,10.27
3,m0,5284,1.8e-07,0,7.74
4,m1,5317,1.8e-07,0,7.79
5,m3,4229,1.8e-07,2,6.19


passage encodé seul ou dans un lot : écart maximal 8.57e-08


**Conclusion.** Six index sont calculés et contrôlés : dimension 384, valeurs finies, normes égales à 1 à 2e-7 près,
entrées identiques de même vecteur. Un passage encodé seul ou dans un lot donne le même vecteur à 1e-7 près : le masque
neutralise bien le remplissage.

**Pour la soutenance.**

- Six index, un par découpage, tous contrôlés : dimension 384, normes à 1, aucune valeur invalide.
- Deux passages au texte identique ont le même vecteur ; un passage a le même vecteur seul ou dans un lot.

**Transition.** Côté Python, tout est cohérent. Reste la question décisive : le navigateur, lui, calcule-t-il comme
Python ? C'est le contrôle 3 (étape 8).

## 8. Contrôle 3 : fidélité de l'encodeur dans le navigateur, et choix de la précision servie

**Objectif.** Mesurer, dans un vrai navigateur, à quel point chaque précision du modèle reproduit le classement de
référence, et choisir celle que le site servira.

**Pourquoi.** C'est le navigateur qui encodera les questions des visiteurs. Une mesure faite seulement en Python
suppose que les deux moteurs calculent pareil. Cette hypothèse s'est révélée fausse.

**Pourquoi ce contrôle existe.** La première version de ce notebook fixait un critère unique au navigateur : un cosinus
d'au moins 0,9999 entre le vecteur calculé par le navigateur et celui du q8 calculé en Python. À la première exécution,
ce critère a **échoué** (cosinus minimal de 0,9942). Le diagnostic, fait dans le navigateur :

- en fp32, le navigateur donne exactement les vecteurs de la référence (cosinus minimal de 1,000000) ;
- la tokenisation de Transformers.js est identique à la nôtre, requête par requête ;
- une moyenne des tokens recalculée à la main redonne exactement le vecteur du navigateur.

Tokenisation, moyenne et normalisation sont donc justes. C'est l'**exécution du fichier q8 par ONNX Runtime Web** qui
diffère de celle d'ONNX Runtime en Python. Le critère reposait sur une hypothèse fausse, que les deux moteurs calculent
pareil. On le remplace par deux contrôles distincts : la fidélité de l'encodeur, mesurée ici **dans le navigateur**, et
l'intégrité de l'export (section 12).

**Rappel : WebAssembly.** WebAssembly (WASM) est un format d'instructions binaire que les navigateurs exécutent, conçu
pour faire tourner efficacement du code compilé. ONNX Runtime Web est compilé dans ce format : ses routines de calcul
ne sont pas celles, optimisées pour le processeur de la machine, qu'ONNX Runtime utilise en Python. Sur un modèle quantifié, où chaque produit passe par des entiers et des arrondis, il est
plausible que cette différence suffise à expliquer l'écart. C'est une hypothèse : le diagnostic isole l'étape en cause, pas le
mécanisme exact.

**Procédure.** La page `scripts/indexation/fidelite_navigateur.html`, servie en local, encode les 822 requêtes de la
section 5 dans le navigateur avec chaque précision du modèle (q8, int8, uint8, fp16, fp32). Ses vecteurs sont
enregistrés dans `controle/fidelite_navigateur.json` et comparés ici à la référence PyTorch fp32, avec l'index fp32.

**Règle de choix, fixée avant la mesure.** Parmi les variantes 8 bits (q8, int8, uint8, 112,6 à 112,8 Mio), on retient
celle qui reproduit le plus souvent le premier résultat de référence ; q8, le choix par défaut de Transformers.js, est
gardé si l'écart n'est pas significatif (McNemar, 5 %). fp16 et fp32 sont mesurés pour situer la perte, pas pour être
servis : ils doublent ou quadruplent le téléchargement du premier chargement (224 et 449 Mio, contre 113 Mio), à
l'opposé de la cible d'environ 50 Mio que le rapport J2 fixe à la première visite (risque R2).

**Le diagnostic, rejoué et vérifié.** La page `scripts/indexation/diagnostic_navigateur.html` isole, sur les 24
requêtes du contrôle 4, les trois étapes de l'encodage dans le navigateur. La cellule suivante vérifie ses trois
conclusions : tokenisation identique à la nôtre, navigateur fp32 identique à la référence, moyenne des tokens
recalculée à la main identique au pipeline q8 du navigateur.

In [11]:
fichier_diag, DIAGNOSTIC = CONTROLE / "diagnostic_navigateur.json", None
fichier_ctrl = CONTROLE / "resultats_navigateur.json"
if fichier_diag.exists() and fichier_ctrl.exists():
    diag = json.loads(fichier_diag.read_text(encoding="utf-8"))
    ctrl = json.loads(fichier_ctrl.read_text(encoding="utf-8"))
    assert diag["questions"] == [r["question"] for r in ctrl["resultats"]], "diagnostic et contrôle sur des requêtes différentes"
    textes_diag = [PREFIXE_REQUETE + q for q in diag["questions"]]
    Q_diag_ref = encoder(textes_diag, "ref", "diagnostic")
    Q_diag_q8 = encoder(textes_diag, "q8", "diagnostic")
    F = np.array(diag["vecteurs_fp32"], dtype=np.float32)
    M = np.array(diag["vecteurs_q8_manuels"], dtype=np.float32)
    P8 = np.array([r["vecteur"] for r in ctrl["resultats"]], dtype=np.float32)

    def cos_lignes(A, B):
        """Cosinus ligne à ligne entre deux matrices de vecteurs."""
        return np.einsum("ij,ij->i", A, B) / np.linalg.norm(A, axis=1) / np.linalg.norm(B, axis=1)

    # trois étapes isolées (tokenisation, calcul fp32, moyenne des tokens), puis l'écart du q8 du navigateur
    DIAGNOSTIC = {
        "tokenisation identique à la nôtre": f"{sum(t == tok.encode(x).ids for t, x in zip(diag['tokens'], textes_diag))} / {len(textes_diag)}",
        "navigateur fp32 contre référence, cosinus minimal": round(float(cos_lignes(F, Q_diag_ref).min()), 7),
        "moyenne manuelle contre pipeline q8 du navigateur, cosinus minimal": round(float(cos_lignes(M, P8).min()), 7),
        "navigateur q8 contre q8 de Python, cosinus minimal": round(float(cos_lignes(P8, Q_diag_q8).min()), 5),
        "q8 de Python contre référence, cosinus minimal": round(float(cos_lignes(Q_diag_q8, Q_diag_ref).min()), 5),
    }
    display(pd.Series(DIAGNOSTIC).to_frame("diagnostic"))
    assert all(t == tok.encode(x).ids for t, x in zip(diag["tokens"], textes_diag))
    assert cos_lignes(F, Q_diag_ref).min() >= SEUIL_PARITE and cos_lignes(M, P8).min() >= SEUIL_PARITE
else:
    print("diagnostic non encore effectué : ouvrir la page de diagnostic, puis réexécuter le notebook")

,diagnostic
tokenisation identique à la nôtre,24 / 24
"navigateur fp32 contre référence, cosinus minimal",1.0
"moyenne manuelle contre pipeline q8 du navigateur, cosinus minimal",1.0
"navigateur q8 contre q8 de Python, cosinus minimal",0.99419
"q8 de Python contre référence, cosinus minimal",0.99866


In [12]:
fichier_fidelite = CONTROLE / "fidelite_navigateur.json"
DTYPE_NAVIGATEUR, FIDELITE = "q8", None
if fichier_fidelite.exists():
    fid = json.loads(fichier_fidelite.read_text(encoding="utf-8"))
    assert fid["requetes"] == TITRES_QUESTIONS and fid["prefixe"] == PREFIXE_REQUETE, "la page a traité d'autres requêtes"
    assert fid["revision"] == MODELE_ONNX["revision"]
    lignes, justes = [], {}
    # pour chaque précision : vecteurs du navigateur comparés à la référence, puis classement sur l'index fp32
    for dtype, res in fid["resultats"].items():
        V = np.array(res["vecteurs"], dtype=np.float32)
        cos_v = np.einsum("ij,ij->i", V, Q_ref)
        S_v, top_v = classement(V, E_ref)
        justes[dtype] = top_v[:, 0] == top_ref[:, 0]
        lignes.append({"précision": dtype, "cosinus avec la référence, minimum": round(float(cos_v.min()), 5),
                       "médiane": round(float(np.median(cos_v)), 5), **accord(top_v, S_v, top_ref),
                       "encodage ms par requête": round(res["encodage_ms"] / len(TITRES_QUESTIONS), 1)})
    FIDELITE = pd.DataFrame(lignes).set_index("précision")
    display(FIDELITE)
    # règle de choix fixée à l'avance
    meilleure = max(PRECISIONS_8_BITS, key=lambda d: justes[d].sum())
    test_dtype = mcnemar_exact(justes[meilleure], justes["q8"]) if meilleure != "q8" else None
    if meilleure != "q8" and test_dtype["p"] < 0.05 and test_dtype["A juste seule"] > test_dtype["B juste seule"]:
        DTYPE_NAVIGATEUR = meilleure
    if meilleure == "q8":
        print("meilleure variante 8 bits : q8, le choix par défaut de Transformers.js ; aucun test nécessaire")
    else:
        print(f"meilleure variante 8 bits : {meilleure} ; test exact de McNemar contre q8 : {test_dtype}")
    # la configuration A reste-t-elle la bonne avec les vrais vecteurs du navigateur ?
    V_nav = np.array(fid["resultats"][DTYPE_NAVIGATEUR]["vecteurs"], dtype=np.float32)
    S_Anav, top_Anav = classement(V_nav, E_ref)
    S_Bnav, top_Bnav = classement(V_nav, E_q8)
    test_ab = mcnemar_exact(top_Anav[:, 0] == top_ref[:, 0], top_Bnav[:, 0] == top_ref[:, 0])
    print(f"avec les vecteurs du navigateur ({DTYPE_NAVIGATEUR}) : A (index fp32) {100 * np.mean(top_Anav[:, 0] == top_ref[:, 0]):.1f} %, "
          f"B (index q8) {100 * np.mean(top_Bnav[:, 0] == top_ref[:, 0]):.1f} % de premier résultat identique ; McNemar {test_ab}")
else:
    print("mesure de fidélité non encore effectuée : ouvrir la page de fidélité, puis réexécuter le notebook ; précision par défaut q8")
print("précision servie au navigateur :", DTYPE_NAVIGATEUR)

,"cosinus avec la référence, minimum",médiane,même premier résultat %,recouvrement du top 5 %,recouvrement du top 10 %,premier de référence dans le top 5 %,"rang du premier de référence, maximum",encodage ms par requête
précision,,,,,,,,
q8,0.99196,0.99655,91.8,90.9,91.3,99.9,9,18.6
int8,0.97326,0.98956,85.8,82.9,83.5,99.3,23,18.1
uint8,0.98262,0.99445,90.8,88.6,89.2,99.9,8,18.2
fp16,1.00000,1.00000,99.9,100.0,100.0,100.0,2,16.7
fp32,1.00000,1.00000,100.0,100.0,100.0,100.0,1,16.6


meilleure variante 8 bits : q8, le choix par défaut de Transformers.js ; aucun test nécessaire


avec les vecteurs du navigateur (q8) : A (index fp32) 91.8 %, B (index q8) 91.4 % de premier résultat identique ; McNemar {'A juste seule': 19, 'B juste seule': 15, 'p': 0.6075913612730801}
précision servie au navigateur : q8


**Conclusion.** Le diagnostic est confirmé : tokenisation identique pour 24 requêtes sur 24, navigateur fp32 identique
à la référence, moyenne manuelle identique au pipeline ; seul le q8 du navigateur s'écarte du q8 de Python (cosinus
minimal de 0,994). Mesuré dans le navigateur, q8 retrouve le premier résultat de référence pour 91,8 % des requêtes, et
ce premier résultat reste dans les 5 premiers pour 99,9 % d'entre elles ; c'est la meilleure des variantes 8 bits, et
elle est servie. Avec ces vrais vecteurs, la configuration A reste la bonne : 91,8 % contre 91,4 % pour B, écart non
significatif (p = 0,61).

**Pour la soutenance.**

- Un critère fixé à l'avance a échoué : on l'a gardé visible, on a cherché la cause, et on a corrigé la méthode.
- Le navigateur en fp32 reproduit exactement la référence ; l'écart vient de l'exécution du q8 par ONNX Runtime Web.
- La précision servie est choisie sur des mesures faites dans le navigateur, par une règle écrite avant la mesure.

**Transition.** Modèle, précision et configuration sont fixés. On peut écrire l'index que le site chargera (étape 9).

## 9. Export de l'index du site

**Objectif.** Écrire l'index sous une forme que le navigateur lit directement, et qui permet de détecter toute
incohérence.

**Pourquoi.** Le site doit charger l'index vite, sans conversion, et ne jamais associer un vecteur au mauvais passage.
Un format simple, accompagné d'empreintes, rend ces deux garanties vérifiables.

**Comment.** Trois fichiers, à copier dans le dossier `public/` du site :

| Fichier | Contenu |
|---|---|
| `index_m2.f32` | la matrice des vecteurs, en float32 petit-boutiste, ligne par ligne : `nb_vecteurs × 384` nombres |
| `passages_m2.json` | un objet par passage, **dans le même ordre que les lignes de la matrice** : identifiant, fiche, section, chemin, liens, date, texte, articles |
| `index_m2.json` | l'en-tête : modèle, révision et précision à charger, préfixe et options de la requête, dimensions, empreintes des deux autres fichiers |

Le site doit lire l'en-tête d'abord et refuser l'index si le nombre de vecteurs, la dimension ou une empreinte ne
correspond pas : c'est la garantie qu'un vecteur ne sera jamais associé au mauvais passage. Côté navigateur,
`new Float32Array(await reponse.arrayBuffer())` relit la matrice sans conversion.

**Rappel : une matrice binaire.** Chaque nombre float32 occupe 4 octets ; « petit-boutiste » signifie que l'octet de
poids faible vient en premier, l'ordre des processeurs x86 et ARM courants, celui que lit `Float32Array`. Les vecteurs
sont rangés ligne par ligne : le vecteur du passage $i$ occupe les octets $4 \times 384 \times i$ à
$4 \times 384 \times (i + 1)$. Il suffit donc de connaître $i$ pour retrouver à la fois le vecteur et le passage.

**Rappel : pourquoi une recherche exhaustive suffit.** Pour une question, le site calcule le produit scalaire avec
chacun des 4 240 vecteurs, soit 4 240 × 384, environ 1,6 million de multiplications, puis trie les scores. Les bases
vectorielles et leurs index approchés servent à éviter ce parcours complet sur des millions de vecteurs ; à cette
échelle, ils n'apportent rien. La durée réelle, mesurée dans le navigateur, figure au contrôle 4.

In [13]:
P_site = TABLES["m2"]
E_site = INDEX[DECOUPAGE_SITE]
chemin_bin, chemin_meta, chemin_entete = SITE / "index_m2.f32", SITE / "passages_m2.json", SITE / "index_m2.json"
# float32 petit-boutiste, ligne par ligne : le vecteur i occupe les octets 4 × 384 × i à 4 × 384 × (i + 1)
E_site.astype("<f4").tofile(chemin_bin)
meta = [{"passage_id": r.passage_id, "titre_fiche": r.titre_fiche, "section_titre": r.section_titre,
         "chemin": list(r.chemin), "url": r.url, "url_citation": r.url_citation, "date_maj": r.date_maj,
         "texte": r.texte, "articles": list(r.articles)} for r in P_site.itertuples()]
chemin_meta.write_text(json.dumps(meta, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
fichier_servi = PRECISIONS_8_BITS[DTYPE_NAVIGATEUR]
# l'en-tête dit au site quel modèle charger, comment encoder la question, et quelles empreintes vérifier
entete_index = {
    "format": "index-e5-v1",
    "modele": {"depot": MODELE_ONNX["depot"], "revision": MODELE_ONNX["revision"], "dtype": DTYPE_NAVIGATEUR,
               "fichier": fichier_servi, "sha256": empreintes_hub[fichier_servi]},
    "requete": {"prefixe": PREFIXE_REQUETE, "pooling": "mean", "normalize": True},
    "index": {"fichier": chemin_bin.name, "type": "float32", "ordre_octets": "little-endian", "disposition": "ligne par ligne",
              "nb_vecteurs": int(E_site.shape[0]), "dimension": DIMENSION, "sha256": sha256_fichier(chemin_bin),
              "moteur": "PyTorch fp32 (intfloat/multilingual-e5-small)" if MOTEUR_INDEX == "ref" else "ONNX q8"},
    "metadonnees": {"fichier": chemin_meta.name, "nb_passages": len(meta), "sha256": sha256_fichier(chemin_meta)},
    "source": {"decoupage": "m2", "empreinte_contenu_notebook_01": manifeste01["methodes"]["m2"]["empreinte_contenu"]},
    "produit_le": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
chemin_entete.write_text(json.dumps(entete_index, ensure_ascii=False, indent=2), encoding="utf-8")

# relecture : ce que le site lira doit redonner exactement ce qui est en mémoire
# relecture : ce que le site lira doit redonner exactement ce qui est en mémoire
relu = np.fromfile(chemin_bin, dtype="<f4").reshape(-1, DIMENSION)
assert np.array_equal(relu, E_site)
meta_relue = json.loads(chemin_meta.read_text(encoding="utf-8"))
assert [m["passage_id"] for m in meta_relue] == P_site.passage_id.tolist()
entete_relu = json.loads(chemin_entete.read_text(encoding="utf-8"))
assert entete_relu["index"]["nb_vecteurs"] == len(meta_relue) == relu.shape[0]
assert entete_relu["index"]["sha256"] == sha256_fichier(chemin_bin) and entete_relu["metadonnees"]["sha256"] == sha256_fichier(chemin_meta)
tailles = pd.DataFrame([{"fichier": c.name, "brut Mio": round(c.stat().st_size / 2**20, 2),
                         "compressé gzip Mio": round(len(gzip.compress(c.read_bytes(), 9)) / 2**20, 2)}
                        for c in (chemin_bin, chemin_meta, chemin_entete)])
display(tailles)
print(f"modèle servi : {fichier_servi} ({DTYPE_NAVIGATEUR})")

,fichier,brut Mio,compressé gzip Mio
0,index_m2.f32,6.21,5.70
1,passages_m2.json,6.13,1.27
2,index_m2.json,0.00,0.00


modèle servi : onnx/model_quantized.onnx (q8)


**Conclusion.** Les trois fichiers sont écrits et relus à l'identique : 6,21 Mio pour la matrice, 6,13 Mio pour les
passages (1,27 Mio une fois compressés), et un en-tête d'environ 1 Kio. Le modèle, lui, n'est pas dans l'export :
Transformers.js le télécharge depuis le Hub, à la révision indiquée par l'en-tête.

**Pour la soutenance.**

- Un format simple et vérifiable : une matrice binaire, un tableau JSON aligné, un en-tête qui porte les empreintes.
- L'export est relu et comparé à la mémoire : ligne `i` de la matrice et passage `i` du JSON sont bien le même passage.
- L'en-tête dit au site quel modèle charger et comment encoder la question : préfixe, moyenne, normalisation, précision.

**Transition.** L'index du site est prêt. Les autres découpages, eux, attendent le notebook 03 : l'étape 10 les exporte
sous une forme qu'il pourra lire.

## 10. Export des index d'évaluation

**Objectif.** Livrer au notebook 03 un index par découpage, prêt à interroger.

**Pourquoi.** Le notebook 03 doit comparer les découpages sur les mêmes questions, sans refaire d'encodage côté
passages. Chaque index doit donc être rattaché sans ambiguïté à sa table de passages.

**Comment.** Chaque découpage est exporté avec la liste ordonnée de ses identifiants de passage. Les métadonnées restent
dans les tables du notebook 01, que l'on retrouve par ces identifiants.

In [14]:
manifeste_variantes = {}
# par découpage : une matrice et la liste ordonnée de ses identifiants, relue pour contrôle
for nom, E in INDEX.items():
    table, colonne = DECOUPAGES[nom]
    chemin_v = VARIANTES / f"{nom}.f32"
    E.astype("<f4").tofile(chemin_v)
    chemin_ids = VARIANTES / f"{nom}.ids.json"
    chemin_ids.write_text(json.dumps(TABLES[table].passage_id.tolist()), encoding="utf-8")
    assert np.array_equal(np.fromfile(chemin_v, dtype="<f4").reshape(-1, DIMENSION), E)
    manifeste_variantes[nom] = {"index": chemin_v.name, "identifiants": chemin_ids.name, "nb_vecteurs": int(E.shape[0]),
                                "table_notebook_01": manifeste01["methodes"][table]["fichier"], "colonne_encodee": colonne,
                                "sha256_index": sha256_fichier(chemin_v)}
(VARIANTES / "variantes.json").write_text(json.dumps(manifeste_variantes, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame(manifeste_variantes).T[["nb_vecteurs", "colonne_encodee", "table_notebook_01"]])

,nb_vecteurs,colonne_encodee,table_notebook_01
m2,4240,entree_encodeur,data/decoupage/passages/passages_m2.parquet
m2_sans_entete,4240,entree_sans_entete,data/decoupage/passages/passages_m2.parquet
m2_256,7009,entree_encodeur,data/decoupage/passages/passages_m2_256.parquet
m0,5284,entree_encodeur,data/decoupage/passages/passages_m0.parquet
m1,5317,entree_encodeur,data/decoupage/passages/passages_m1.parquet
m3,4229,entree_encodeur,data/decoupage/passages/passages_m3.parquet


**Conclusion.** Six index d'évaluation sont écrits dans `data/indexation/variantes/` (M2, M2 sans en-tête, M2 à 256
tokens, M0, M1 et M3), chacun avec ses identifiants et son empreinte.

**Pour la soutenance.**

- Le notebook 03 n'aura rien à encoder côté passages : il comparera les découpages sur le jeu de questions, à partir de ces index.
- Les questions du jeu d'évaluation devront, elles, être encodées dans le navigateur, avec la précision servie.

**Transition.** Avant le contrôle final, un premier regard sur ce que l'index retrouve, avec quelques requêtes
d'exemple (étape 11).

## 11. Exemples de recherche

**Objectif.** Montrer, sur quelques requêtes, que l'index renvoie des résultats classés et plausibles.

**Pourquoi.** Le ticket #30 demande que plusieurs requêtes renvoient des résultats classés. C'est aussi le moyen de
repérer des comportements à surveiller dans l'évaluation.

**Comment.** Les quatre requêtes du support pédagogique, sur l'index du site. Ce sont des **illustrations** : le support
en commente les réussites et l'échec (la période d'essai d'un CDI), examiné juste après. Les requêtes sont encodées ici
en fp32 ; la section 12 rejoue des requêtes dans le navigateur, avec la précision servie.

**Rappel : lire un score.** Les scores de ce modèle se concentrent entre 0,7 et 1,0, à cause de la température de 0,01
utilisée à son entraînement (FAQ de la carte du modèle). Un score de 0,88 ne signifie donc pas « très proche » : seul
compte l'ordre des scores pour une même question.

In [15]:
Q_ex = encoder([PREFIXE_REQUETE + q for q in REQUETES_EXEMPLE], "ref", "exemples")
S_ex, top_ex = classement(Q_ex, E_site, k=3)
lignes = []
for i, q in enumerate(REQUETES_EXEMPLE):
    for rang, j in enumerate(top_ex[i], 1):
        r = P_site.iloc[j]
        lignes.append({"requête": q if rang == 1 else "", "rang": rang, "score": round(float(S_ex[i, j]), 4),
                       "fiche > section": f"{r.titre_fiche[:40]} > {r.section_titre[:45]}", "lien": r.url_citation[:70]})
display(pd.DataFrame(lignes))

,requête,rang,score,fiche > section,lien
0,Quelles mentions sont interdites sur le bulletin de paie ?,1,0.9022,Le bulletin de paie > Et les mentions interdites ?,https://travail-emploi.gouv.fr/le-bulletin-de-paie#:~:text=Aucune%20me
1,,2,0.8927,Le bulletin de paie > Chapô,https://travail-emploi.gouv.fr/le-bulletin-de-paie#:~:text=Au%20moment
2,,3,0.8839,Le bulletin de paie > Quelles sont les mentions obligatoires ?,https://travail-emploi.gouv.fr/le-bulletin-de-paie#:~:text=Les%20menti
3,Mon employeur peut-il me licencier pendant un arrêt maladie ?,1,0.8846,Les absences liées à la maladie ou à l'a > Peut-il y avoir licenciement pour maladie ?,https://travail-emploi.gouv.fr/les-absences-liees-la-maladie-ou-laccid
4,,2,0.8835,Les absences liées à la maladie ou à l'a > Chapô,https://travail-emploi.gouv.fr/les-absences-liees-la-maladie-ou-laccid
5,,3,0.8796,Les absences liées à la maladie ou à l'a > Quels sont les droits et obligations du salar,https://travail-emploi.gouv.fr/les-absences-liees-la-maladie-ou-laccid
6,Combien de temps peut durer la période d'essai d'un CDI ?,1,0.8927,Le contrat à durée déterminée (CDD) > Quelle est la durée de la période d'essai ?,https://travail-emploi.gouv.fr/le-contrat-duree-determinee-cdd#:~:text
7,,2,0.8873,CDD multi-remplacement : relance de l'ex > Terme du contrat,https://travail-emploi.gouv.fr/cdd-multi-remplacement-relance-de-lexpe
8,,3,0.8854,La période de reconversion > Que se passe-t-il au terme de la période d’es,https://travail-emploi.gouv.fr/la-periode-de-reconversion#:~:text=Si%2
9,Peut-on être viré quand on est en arrêt de travail ?,1,0.8764,Les absences liées à la maladie ou à l'a > Chapô,https://travail-emploi.gouv.fr/les-absences-liees-la-maladie-ou-laccid


**L'échec, examiné.** Pour la période d'essai d'un CDI, les trois premiers passages portent sur le CDD ou sur d'autres
contrats, alors que la fiche « La période d'essai » répond à la question, et plus précisément son passage qui donne les
durées maximales du CDI, selon l'article L. 1221-19. On cherche le rang de la fiche (son premier passage) et celui de ce
passage, avec le sigle, puis avec le terme développé. Une seule requête : c'est une piste pour le jeu de questions du
notebook 03 (y inclure des sigles), pas une mesure.

In [16]:
FICHE_ATTENDUE = "La période d'essai"
VARIANTES_CDI = ["Combien de temps peut durer la période d'essai d'un CDI ?",
                 "Combien de temps peut durer la période d'essai d'un contrat à durée indéterminée ?"]
Q_cdi = encoder([PREFIXE_REQUETE + q for q in VARIANTES_CDI], "ref", "exemple_cdi")
S_cdi, ordre_cdi = classement(Q_cdi, E_site, k=len(P_site))
dans_fiche = P_site.titre_fiche.to_numpy() == FICHE_ATTENDUE
lignes_attendues = np.flatnonzero(dans_fiche)
# trois passages de la fiche citent l'article ; celui qui donne les durées maximales du CDI commence par cette phrase
DEBUT_REPONSE = "Au terme de l’article L. 1221-19 du Code du travail, le contrat de travail à durée indéterminée"
passage_reponse = np.flatnonzero(dans_fiche & P_site.texte.str.startswith(DEBUT_REPONSE).to_numpy())
assert len(lignes_attendues) > 0 and len(passage_reponse) == 1, (len(lignes_attendues), len(passage_reponse))
for i, (q, ordre) in enumerate(zip(VARIANTES_CDI, ordre_cdi)):
    rang_fiche = 1 + int(np.flatnonzero(np.isin(ordre, lignes_attendues))[0])
    rang_passage = 1 + int(np.flatnonzero(ordre == passage_reponse[0])[0])
    ecart = S_cdi[i, ordre[0]] - S_cdi[i, passage_reponse[0]]
    print(f"{q}\n   rang de la fiche : {rang_fiche} | rang du passage L. 1221-19 : {rang_passage}, à {ecart:.4f} du premier "
          f"| premier résultat : {P_site.titre_fiche.iloc[ordre[0]][:45]}")

Combien de temps peut durer la période d'essai d'un CDI ?
   rang de la fiche : 4 | rang du passage L. 1221-19 : 5, à 0.0089 du premier | premier résultat : Le contrat à durée déterminée (CDD)
Combien de temps peut durer la période d'essai d'un contrat à durée indéterminée ?
   rang de la fiche : 1 | rang du passage L. 1221-19 : 2, à 0.0025 du premier | premier résultat : La période d'essai


**Conclusion.** Les quatre requêtes renvoient des passages classés. Trois d'entre elles placent en tête des passages sur
le bon sujet, y compris la requête familière « Peut-on être viré... », alors que le mot « viré » n'apparaît dans aucun
passage (mesure du support pédagogique). La requête sur le CDI échoue : avec « CDI », la
fiche attendue n'arrive qu'au rang 4 et le passage des durées au rang 5 ; avec « contrat à durée indéterminée », la
fiche passe au rang 1 et ce passage au rang 2. Le modèle rapproche mal le sigle de sa forme développée, du moins sur cet
exemple.

**Pour la soutenance.**

- Les exemples sont des illustrations, pas une évaluation : quatre requêtes ne mesurent pas un système.
- L'échec sur « CDI » est une piste concrète : le jeu de questions doit contenir des sigles, et l'extension des sigles
  de la question est une amélioration à mesurer.

**Transition.** Dernière vérification avant de livrer : le site, qui lira ces fichiers dans le navigateur, retrouve-t-il
exactement les classements de Python ? C'est le contrôle 4 (étape 12).

## 12. Contrôle 4 : intégrité de l'export, vérifiée dans le navigateur

**Objectif.** Vérifier que le site, qui lit l'en-tête, la matrice binaire et le JSON, classe les passages exactement
comme Python le ferait sur ces mêmes fichiers.

**Pourquoi.** Entre Python et le navigateur, beaucoup de choses peuvent se perdre : l'ordre des octets, l'alignement du
JSON sur la matrice, le modèle ou les options lus dans l'en-tête. Ce contrôle les vérifie toutes en une fois. Il ne
dépend pas de la précision du modèle : pour chaque requête, Python recalcule le classement **avec le vecteur produit par
le navigateur**.

**Comment.** Cette cellule écrit `controle/attendu.json` (24 requêtes). La page
`scripts/indexation/controle_navigateur.html`, servie en local depuis la racine du dépôt, lit l'export du site, en
vérifie les empreintes, charge le modèle indiqué par l'en-tête, encode les requêtes et calcule les 5 premiers passages.
Ses résultats sont enregistrés dans `controle/resultats_navigateur.json`.

**Critères fixés à l'avance.** Le navigateur a chargé la précision de l'en-tête ; les empreintes SHA-256 qu'il calcule
sur la matrice et sur le JSON reçus sont celles de l'en-tête ; pour chaque requête, ses 5 premiers passages sont ceux
que Python calcule avec son vecteur, dans le même ordre, à des ex aequo près (scores séparés de moins de 1e-6).

**Rappel : l'API Web Crypto.** La page calcule les empreintes avec `crypto.subtle.digest("SHA-256", octets)`, l'API
cryptographique standard des navigateurs. Elle n'est disponible que dans un contexte sécurisé : une page servie en
HTTPS, ou en local depuis `127.0.0.1` ou `localhost`, comme ici.

In [17]:
rng = np.random.default_rng(GRAINE + 1)
requetes_nav = REQUETES_EXEMPLE + [str(t) for t in rng.choice(TITRES_QUESTIONS, N_REQUETES_NAVIGATEUR, replace=False)]
attendu = {"requetes": [{"question": q} for q in requetes_nav], "entete_index": entete_index}
(CONTROLE / "attendu.json").write_text(json.dumps(attendu, ensure_ascii=False), encoding="utf-8")
print(f"attendu.json écrit : {len(requetes_nav)} requêtes")

resultats_nav = CONTROLE / "resultats_navigateur.json"
CONTROLE_NAVIGATEUR = None
if resultats_nav.exists():
    nav = json.loads(resultats_nav.read_text(encoding="utf-8"))
    assert [r["question"] for r in nav["resultats"]] == requetes_nav, "le navigateur a traité d'autres requêtes"
    assert nav.get("dtype") == DTYPE_NAVIGATEUR, f"le navigateur a chargé {nav.get('dtype')}, l'en-tête indique {DTYPE_NAVIGATEUR}"
    # empreintes calculées par la page sur les octets reçus : le navigateur a lu exactement les fichiers exportés
    assert nav.get("sha256_index") == entete_index["index"]["sha256"], "le navigateur a lu un autre index que celui exporté"
    assert nav.get("sha256_metadonnees") == entete_index["metadonnees"]["sha256"], "le navigateur a lu d'autres métadonnées"
    identiques, ecarts = 0, []
    # pour chaque requête : classement recalculé par Python avec le vecteur du navigateur, comparé à celui du navigateur
    for i, r in enumerate(nav["resultats"]):
        v = np.array(r["vecteur"], dtype=np.float32)
        s_py = E_site @ v
        attendu_top = np.argsort(-s_py, kind="stable")[:5]
        nav_top = [P_site.index[P_site.passage_id == pid][0] for pid in r["top5"]]
        # ex aequo : un écart d'ordre n'est accepté que si les scores concernés sont égaux à la tolérance près
        ok = all(a == b or abs(s_py[a] - s_py[b]) < TOLERANCE_EGALITE for a, b in zip(attendu_top, nav_top))
        identiques += ok
        ecarts.append(float(np.abs(np.array(r["scores5"]) - s_py[nav_top]).max()))
    cos_python_q8 = None
    if DTYPE_NAVIGATEUR == "q8":
        V_ctrl = np.array([r["vecteur"] for r in nav["resultats"]], dtype=np.float32)
        Q_ctrl = encoder([PREFIXE_REQUETE + q for q in requetes_nav], "q8", "navigateur")
        cos_python_q8 = float(np.einsum("ij,ij->i", V_ctrl, Q_ctrl).min())
    CONTROLE_NAVIGATEUR = {"requetes": len(requetes_nav), "précision chargée": nav["dtype"],
                           "top 5 identique à celui de Python": f"{identiques} / {len(requetes_nav)}",
                           "écart maximal entre scores du navigateur et de Python": f"{max(ecarts):.1e}",
                           "cosinus minimal avec le q8 de Python (information)": None if cos_python_q8 is None else round(cos_python_q8, 5),
                           "recherche exhaustive et tri, ms par requête (médiane)":
                               round(float(np.median([r["recherche_ms"] for r in nav["resultats"]])), 1),
                           "Transformers.js": nav.get("transformers_js"), "durée totale ms": round(nav.get("duree_ms", 0))}
    display(pd.Series(CONTROLE_NAVIGATEUR).to_frame("contrôle navigateur"))
    assert identiques == len(requetes_nav), "le navigateur ne classe pas les passages comme Python sur l'export"
else:
    print("contrôle d'intégrité non encore effectué : ouvrir la page de contrôle, puis réexécuter le notebook")

attendu.json écrit : 24 requêtes


,contrôle navigateur
requetes,24
précision chargée,q8
top 5 identique à celui de Python,24 / 24
écart maximal entre scores du navigateur et de Python,6.0e-08
cosinus minimal avec le q8 de Python (information),0.99419
"recherche exhaustive et tri, ms par requête (médiane)",1.7
Transformers.js,4.3.0
durée totale ms,1287


**Conclusion.** Contrôle 4 réussi : le navigateur a chargé le modèle q8 indiqué par l'en-tête, les empreintes qu'il
calcule sont celles de l'export, et ses 5 premiers passages sont ceux de Python pour 24 requêtes sur 24, à 6e-8 près
sur les scores. La recherche exhaustive et le tri prennent 1,7 ms par question en médiane, dans le navigateur.

**Pour la soutenance.**

- Le dernier contrôle se fait dans un vrai navigateur, sur les fichiers exportés : c'est la chaîne que le site exécutera.
- Il isole l'intégrité de l'export (octets, alignement, en-tête) de la précision du modèle, mesurée en section 8.

**Transition.** Tous les contrôles sont passés. Il reste à tout consigner dans un manifeste (étape 13).

## 13. Manifeste et synthèse

**Objectif.** Consigner tout ce qui permet de reproduire ou de contrôler l'indexation, et résumer les résultats.

**Pourquoi.** Dans quelques semaines, il faudra pouvoir dire exactement avec quels fichiers, quels modèles et quelles
versions l'index du site a été produit, et avec quels résultats de contrôle.

**Comment.** Le manifeste fixe les entrées et leurs empreintes, les modèles et révisions, les empreintes des fichiers
ONNX, la configuration retenue, les résultats des contrôles, les fichiers produits et les durées d'encodage. Deux
exécutions doivent produire des index identiques octet pour octet.

In [18]:
# tout ce qui permet de rejouer ou de contrôler l'indexation, écrit à côté des index
manifeste02 = {
    "notebook": "scripts/indexation/02_indexation.ipynb", "execute_le": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "entrees": {code: manifeste01["methodes"][code] for code in TABLES},
    "modeles": {"reference": MODELE_REF, "onnx": {**MODELE_ONNX, "fichiers": FICHIERS_ONNX}},
    "controles": {
        "parite_cosinus_minimal": float(cos_parite.min()), "parite_seuil": SEUIL_PARITE,
        "quantification_python": tableau_quant.to_dict(orient="index"), "mcnemar_python": test, "configuration": CONFIGURATION,
        "fidelite_navigateur": None if FIDELITE is None else FIDELITE.to_dict(orient="index"),
        "precision_servie": DTYPE_NAVIGATEUR, "integrite_navigateur": CONTROLE_NAVIGATEUR,
    },
    "site": entete_index, "variantes": manifeste_variantes, "versions": VERSIONS,
    "durees_encodage_s": {f"{n}/{m}": round(d, 1) for (n, m), d in DUREES.items()},
}
(SORTIES / "manifeste.json").write_text(json.dumps(manifeste02, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
synthese = pd.Series({
    "passages indexés pour le site": f"{E_site.shape[0]} × {DIMENSION}",
    "parité ONNX fp32 et référence, cosinus minimal": f"{cos_parite.min():.7f}",
    "configuration retenue": f"{CONFIGURATION} ({'index fp32, requête q8' if CONFIGURATION == 'A' else 'index q8, requête q8'})",
    "précision servie au navigateur": DTYPE_NAVIGATEUR,
    "même premier résultat qu'en fp32 intégral, mesuré dans le navigateur":
        f"{FIDELITE.loc[DTYPE_NAVIGATEUR, 'même premier résultat %']} %" if FIDELITE is not None else "mesure à effectuer",
    "même premier résultat, simulation Python du q8": f"{tableau_quant.loc['A : index fp32, requête q8', 'même premier résultat %']} %",
    "intégrité de l'export dans le navigateur": CONTROLE_NAVIGATEUR["top 5 identique à celui de Python"] if CONTROLE_NAVIGATEUR else "à effectuer",
    "index d'évaluation exportés": ", ".join(INDEX),
})
display(synthese.to_frame("synthèse"))

,synthèse
passages indexés pour le site,4240 × 384
"parité ONNX fp32 et référence, cosinus minimal",0.9999998
configuration retenue,"A (index fp32, requête q8)"
précision servie au navigateur,q8
"même premier résultat qu'en fp32 intégral, mesuré dans le navigateur",91.8 %
"même premier résultat, simulation Python du q8",96.5 %
intégrité de l'export dans le navigateur,24 / 24
index d'évaluation exportés,"m2, m2_sans_entete, m2_256, m0, m1, m3"


**Conclusion.** La synthèse ci-dessus résume la chaîne : 4 240 passages indexés, parité du modèle ONNX, configuration A,
précision q8 servie, fidélité mesurée dans le navigateur et intégrité de l'export vérifiée par le navigateur lui-même.

**Pour la soutenance.**

- Le manifeste permet de rejouer l'indexation et d'en contrôler chaque fichier par son empreinte.
- Deux exécutions complètes sans cache, comparées le 27/09/2026 hors du notebook, ont produit des index identiques
  octet pour octet (documentation du notebook 02).

## Bilan : ce que ce notebook établit, et ce qu'il n'établit pas

**Établi par les cellules ci-dessus** : les entrées sont celles du notebook 01 ; le modèle ONNX fp32 calcule les mêmes
vecteurs que la référence, en Python comme dans le navigateur ; l'effet de la quantification sur le classement est
mesuré dans le navigateur, qui fait foi, et la précision servie est choisie par une règle fixée à l'avance ; l'export du
site est relu à l'identique, et le navigateur classe les passages exactement comme Python sur ces mêmes fichiers.

**Non établi** : la qualité de la recherche et la comparaison des découpages. Il faut pour cela le jeu de questions
annotées ; ce sera le notebook 03, qui partira des index de `data/indexation/variantes/`.

## Et ensuite : le notebook 03, évaluation de la recherche

**Objectif.** Mesurer enfin la qualité de la recherche, ce que les notebooks 01 et 02 ne font pas : pour une vraie
question, le système trouve-t-il le passage qui répond, et à quel rang ?

**Pourquoi il n'existe pas encore.** Il lui faut un **jeu de questions annotées**, en préparation par Jibril et
Maïmouna. Le rapport J2 prévoit qu'il soit rédigé par des membres qui ne construisent pas l'index (risque R7) : sinon,
les questions ressembleraient au texte des passages et la mesure serait flattée, comme avec nos titres-questions de
contrôle. Chaque question sera annotée par l'URL de la fiche et l'extrait qui répond, indépendamment de tout découpage :
c'est ce qui permet de juger M0, M1, M2 et M3 sur la même vérité terrain. La règle qui déclare un passage pertinent
(contient-il l'extrait entier, ou une part fixée à l'avance ?) devra être écrite avant la mesure, parce que M1 et M3
peuvent couper un extrait entre deux passages. Une part de questions hors corpus, 30 % d'après le J2, servira au volet
B, qui doit savoir s'abstenir.

**Les métriques retenues au J2.** Pour un ensemble $Q$ de questions :

- **Recall@k**, pour $k$ valant 1, 3, 5 et 10 : la part des questions dont au moins un passage pertinent figure parmi
  les $k$ premiers, $\text{Recall@}k = \frac{1}{|Q|} \sum_{q \in Q} \mathbf{1}[\text{un passage pertinent est dans les } k \text{ premiers}]$ ;
- **MRR** (rang réciproque moyen) : $\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{r_q}$, où $r_q$ est le rang du
  premier passage pertinent. Un bon résultat au rang 1 vaut 1, au rang 5 vaut 0,2 ; on fixe une coupure (par exemple
  10) au-delà de laquelle la question compte 0.

**Les comparaisons prévues**, toutes appariées (mêmes questions pour chaque système, test de McNemar) :

1. **les découpages** : M2 contre M0, M1 et M3. M2 reste la méthode de travail, sauf si une autre fait significativement
   mieux ;
2. **les variantes de M2** : budget de 256 tokens, et passages sans en-tête, pour mesurer l'effet de chacun ;
3. **les références lexicales** : tirage aléatoire, TF-IDF et BM25. Le critère de réussite du J2 est que l'encodeur ne
   fasse pas moins bien que BM25. C'est là que se posera la question des mots vides et de la racinisation, étudiée au
   notebook 01 ;
4. **la précision de la requête** : questions encodées dans le navigateur en q8, la précision servie, et en fp32, pour
   mesurer l'effet réel du q8 sur la qualité, et non plus seulement sur le classement ;
5. **les sigles** : un sous-ensemble de questions avec sigles (CDI, CDD, RTT, CSE, comme l'a proposé Remy), et l'effet
   d'une extension des sigles de la question, suggérée par l'échec de la section 11.

**Ce qui conditionne la portée des résultats.** Le dimensionnement du J2 compte : en comparaison appariée, avec un taux
de désaccord de 0,20 entre deux systèmes, détecter un écart de 10 points demande environ 155 questions, et un écart de
15 points environ 68. L'échéancier du J2 prévoit 60 questions : de quoi détecter seulement de grands écarts. Le nombre
de questions est donc à trancher avec l'équipe. Deux autres garde-fous sont prévus au J2 : un accord entre annotateurs
mesuré sur un échantillon, et une validation préalable de la chaîne de mesure sur le jeu public PIAF, que prépare Remy.

**Aucun seuil absolu.** Le rapport technique de mE5 publie, pour `multilingual-e5-small`, un nDCG@10 de 47,6 sur la
partie française de MIRACL (tableau 6), contrairement à ce qu'indiquait le rapport J2 ; mais MIRACL porte sur Wikipédia,
pas sur le droit du travail. Les résultats du notebook 03 s'exprimeront donc en écarts appariés face aux références, et
le seuil d'affichage ou d'abstention se calibrera sur le jeu de questions.

**Ce que le notebook 03 lira.** Les index de `data/indexation/variantes/` et leurs identifiants, les tables du
notebook 01 pour les métadonnées, et les vecteurs des questions, encodés dans le navigateur par une page du même type
que `fidelite_navigateur.html`.